# Machine Failure Risk Prediction

This project builds an end-to-end supervised-learning workflow for machine-failure classification using the **UCI AI4I 2020 Predictive Maintenance Dataset**.

The notebook is the narrative analysis layer of the repository. Reusable implementation lives in the `predictive_maintenance` package:

- `data.py` — loading, schema extraction, and train/test splitting
- `data_quality.py` — validation checks
- `eda_visualization.py` — exploratory plots
- `processing.py` — feature engineering and preprocessing
- `models.py` — candidate models and hyperparameter searches
- `evaluation.py` — cross-validation, threshold selection, calibration, cost analysis, and final metrics
- `interpretability.py` — post-hoc explanations and error analysis
- `plotting.py` — evaluation and interpretation figures
- `reporting.py` — CSV reports, confidence-interval tables, summaries, and model artifacts

# 1. Problem Definition

## Objective

Predict whether a machine will fail from a snapshot of its operating conditions:

- air temperature
- process temperature
- rotational speed
- torque
- tool wear
- product type

The task is strongly imbalanced, so the workflow emphasizes **Average Precision**, recall, F2-score, threshold selection and operational error costs rather than accuracy alone.

## Operational meaning of errors

A **false negative** is a missed failure. It can represent unplanned downtime, production interruption or equipment damage.

A **false positive** is an unnecessary maintenance alert. It consumes inspection time but is generally less costly than an undetected failure.

The final operating threshold is therefore selected from training-set out-of-fold predictions by maximizing precision while satisfying a minimum recall requirement.

## Synthetic-data limitation

AI4I 2020 is a synthetic benchmark generated from documented operating rules and stochastic processes. It is useful for demonstrating a rigorous ML workflow but its absolute performance must not be interpreted as evidence of production readiness.

Any real deployment would require temporal validation, machine-specific data, calibrated operating costs and monitoring for distribution shift.

# 2. Environment and Package Imports

Install the repository in editable mode before running the notebook:

```bash
pip install -e ".[notebook,dev]"
```

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import cross_validate
from predictive_maintenance import config, data, data_quality, eda_visualization, evaluation, interpretability, models, plotting, processing, reporting
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

reports_dir = project_root / "reports" / "notebook"
artifacts_dir = project_root / "artifacts"
tables_dir, figures_dir, artifacts_dir = reporting.create_output_directories(reports_dir, artifacts_dir)

In [ ]:
print(f"RANDOM_STATE = {config.RANDOM_STATE}")
print(f"TEST_SIZE = {config.TEST_SIZE}")
print(f"N_SPLITS = {config.N_SPLITS}")
print(f"MIN_RECALL = {config.MIN_RECALL}")

# 3. Data Loading and Schema

In [ ]:
maintenance_df, dataset = data.load_raw_dataset()
print(f"Dataset shape: {maintenance_df.shape}")
display(dataset.variables)

## Feature and target policy

`UDI` and `Product ID` are identifiers rather than operational predictors and are excluded from modeling.

`TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are failure-mode flags used to construct the binary `Machine failure` target. They are never used as model inputs. They are retained only for post-hoc failure-mode analysis after the model and threshold are frozen.

In [ ]:
identifier_columns = {"UDI", "Product ID"}
feature_columns, target_columns, numerical_features, categorical_features = data.get_schema(dataset)

feature_columns = [column for column in feature_columns if column not in identifier_columns]
numerical_features = [column for column in numerical_features if column not in identifier_columns]
categorical_features = [column for column in categorical_features if column not in identifier_columns]

feature_data, target_data = data.get_features_and_target(maintenance_df, feature_columns)

print(f"Model-input features: {list(feature_columns)}")
print(f"Target columns in source data: {list(target_columns)}")
print(f"Feature matrix shape: {feature_data.shape}")
print(f"Target shape: {target_data.shape}")
display(feature_data.head())

## Target construction

`Machine failure` equals 1 when at least one documented failure mechanism is active.

The mechanisms are separate but not necessarily statistically independent:

- **TWF:** stochastic tool-wear failure within a specified wear interval
- **HDF:** heat-dissipation failure based on temperature difference and rotational speed
- **PWF:** power failure based on mechanical power
- **OSF:** overstrain failure based on torque, tool wear, and product type
- **RNF:** random failure not recoverable from the operating predictors

HDF, PWF, and OSF are strongly recoverable from the available inputs because their rules use those inputs directly. TWF is partially predictable from tool wear, while RNF is irreducible from the provided features.

In [ ]:
target_integrity = (maintenance_df[config.FAILURE_MODE_COLUMNS].max(axis=1) == maintenance_df["Machine failure"])
print(f"Rows consistent with failure-mode OR rule: {target_integrity.mean():.2%}")

failure_overlap = maintenance_df[config.FAILURE_MODE_COLUMNS].sum(axis=1).value_counts().sort_index()
display(failure_overlap.rename("Row count").to_frame())

The consistency rate should be at (or extremely close to) 100% — `Machine failure` is defined as the logical OR of the five failure-mode flags, so any deviation would mean either a labeling anomaly in this data pull or a misreading of the construction rule and should be resolved before trusting anything downstream.

`failure_overlap` counts how many of the five mechanisms are simultaneously active per row. Rows with more than one active mechanism matter for the Failure-Mode Detection analysis in Section 12: a false negative on such a row can't be attributed to a single failure mode, since more than one documented cause was present.

# 4. Data Quality

In [ ]:
missing_report = data_quality.missing_values_report(maintenance_df)
duplicate_report = data_quality.duplicate_check(maintenance_df, feature_data)
class_report = data_quality.class_distribution(target_data)
range_report = data_quality.range_validation(feature_data, numerical_features)
temperature_violations = data_quality.process_temperature_check(feature_data)

display(missing_report)
display(pd.DataFrame([duplicate_report]))
display(class_report.round(3))
display(range_report.round(3))
print(f"Rows where Process temperature < Air temperature: {temperature_violations}")

In [ ]:
reporting.save_table(missing_report, tables_dir / "missing_values.csv")
reporting.save_table(pd.DataFrame([duplicate_report]), tables_dir / "duplicate_report.csv", include_index=False)
reporting.save_table(class_report, tables_dir / "class_distribution.csv")
reporting.save_table(range_report, tables_dir / "range_validation.csv")

## Data-quality interpretation

The checks above should confirm:

- no missing sensor or target values
- no physically impossible negative values in the numeric operating columns
- no process-temperature values below air temperature
- a strongly imbalanced target distribution
- any exact duplicates requiring investigation before modeling

Because this dataset is synthetic, these checks primarily protect against loading errors or future source changes.

# 5. Exploratory Analysis

Class-conditional distributions are normalized independently so that the minority failure class remains visible.

In [ ]:
feature_units = {"Torque": "Nm", "Rotational speed": "rpm", "Tool wear": "min", "Process temperature": "K", "Air temperature": "K"}

for feature, unit in feature_units.items():
    histogram_figure, boxplot_figure = eda_visualization.plot_normalized_distribution(maintenance_df, feature, unit)
    display(histogram_figure, boxplot_figure)
    plotting.save_figure(histogram_figure,figures_dir/f"{feature}_histogram.png")
    plotting.save_figure(boxplot_figure,figures_dir/f"{feature}_boxplot.png")

## Product-type failure rate

Type L has the lowest documented OSF threshold and is therefore easier to push into overstrain failure under otherwise similar operating conditions. Type H has the highest threshold.

In [ ]:
type_figure, failure_rate_by_type = eda_visualization.plot_failure_rate_by_type(maintenance_df)
display(type_figure)
display(failure_rate_by_type.round(3).rename("Failure rate (%)").to_frame())
plotting.save_figure(type_figure,figures_dir/f"Failure_rate_by_product_type.png")

## Domain-informed interactions

The following views are tied to documented failure mechanisms rather than arbitrary pairwise plots:

- torque versus rotational speed for power-related behavior
- mechanical-power estimate for PWF
- process-minus-air temperature gap versus speed for HDF

In [ ]:
torque_vs_rotational_figure = eda_visualization.plot_torque_vs_speed(maintenance_df)
display(torque_vs_rotational_figure)
plotting.save_figure(torque_vs_rotational_figure,figures_dir/"torque_vs_rotational.png")

estimated_power_figure = eda_visualization.plot_power_estimate(maintenance_df)
display(estimated_power_figure)
plotting.save_figure(estimated_power_figure,figures_dir/"estimated_power_figure.png")

temperature_vs_rotational_figure = eda_visualization.plot_temperature_gap_vs_speed(maintenance_df)
display(temperature_vs_rotational_figure)
plotting.save_figure(temperature_vs_rotational_figure,figures_dir/"temperature_vs_rotational")

## EDA Findings

`Torque` and `Rotational speed` show the clearest separation between the classes. Failures generally occur at higher torque and lower rotational speed, although a smaller failure group also appears at very low torque and unusually high speed. This indicates that failures are associated with extreme operating combinations rather than a single linear pattern.

The estimated-power plot supports this observation: failures are concentrated below the documented 3,500 W boundary and above 9,000 W, motivating the engineered `Power` feature. `Tool wear` is also higher for failures, with a visible concentration around 190–220 minutes, supporting both wear-related effects and the `Torque x Tool wear` interaction.

Air and process temperatures individually show substantial class overlap. However, failures are visible where the process-to-air temperature difference is below approximately 8.6 K and rotational speed is below 1,380 rpm. This motivates the engineered `Temperature difference` feature.

Failure rates also differ by product type: Type L has the highest observed rate (3.92%), followed by M (2.77%) and H (2.09%).

Overall, the EDA supports multivariate modeling because no single feature fully separates failures from non-failures. These findings are descriptive and specific to this synthetic benchmark.

# 6. Train/Test Split and Feature Engineering

The holdout test set is created before feature engineering and is not used for model selection, hyperparameter tuning, calibration analysis or threshold selection.

In [ ]:
feature_data_train, feature_data_test, target_data_train, target_data_test = data.split_train_test(feature_data, target_data)

split_distribution = pd.DataFrame({"Training": target_data_train.value_counts(normalize=True), "Test": target_data_test.value_counts(normalize=True)})
display(split_distribution.round(4))

## Engineered features

Three domain-informed interactions are added:

- `Power` = torque × angular velocity
- `Temperature difference` = process temperature − air temperature
- `Torque x Tool wear` = torque × tool wear

These interactions reflect the documented PWF, HDF and OSF mechanisms while remaining valid transformations of the permitted input features.

In [ ]:
feature_data_train = processing.add_engineered_features(feature_data_train)
feature_data_test = processing.add_engineered_features(feature_data_test)
engineered_numerical_features = processing.get_engineered_numerical_features(numerical_features)

display(feature_data_train[["Power", "Temperature difference", "Torque x Tool wear"]].describe().round(3))

## Raw-versus-engineered ablation

The same XGBoost configuration is evaluated on identical folds with and without the engineered features. Average Precision is the primary metric. F2 is included because missed failures are costly.

In [ ]:
ablation_cv = evaluation.make_cv()
raw_columns = list(numerical_features) + list(categorical_features)
engineered_columns = engineered_numerical_features + list(categorical_features)
ablation_scoring = {"average_precision": "average_precision", "f2": evaluation.CV_SCORING["f2"]}
ablation_results = []
ablation_fold_scores = {}

for label, numeric_columns, all_columns in [("Raw features only", list(numerical_features), raw_columns), ("Raw + engineered features", engineered_numerical_features, engineered_columns)]:
    linear_ablation_preprocessor, tree_ablation_preprocessor = processing.build_preprocessors(numeric_columns, categorical_features)
    ablation_models = models.build_models(tree_ablation_preprocessor, linear_ablation_preprocessor, scale_pos_weight=1.0)
    scores = cross_validate(ablation_models["XGBoost"], feature_data_train[all_columns], target_data_train, cv=ablation_cv, scoring=ablation_scoring, n_jobs=-1)
    ablation_fold_scores[label] = scores["test_average_precision"]
    ablation_results.append({"Feature set": label, "CV Average Precision": scores["test_average_precision"].mean(), "CV Average Precision Std": scores["test_average_precision"].std(), "CV F2": scores["test_f2"].mean()})

ablation_results = pd.DataFrame(ablation_results)
display(ablation_results.round(4))

In [ ]:
paired_ap_difference = ablation_fold_scores["Raw + engineered features"] - ablation_fold_scores["Raw features only"]
display(pd.Series(paired_ap_difference, name="Engineered minus raw AP by fold").to_frame().round(4))
print(f"Mean paired AP difference: {paired_ap_difference.mean():.4f}")

Interpret the paired fold differences rather than treating one model's fold standard deviation as a formal significance threshold. Retain the engineered features when they improve performance consistently or provide a defensible domain representation without materially degrading validation results.

# 7. Experimental Design

In [ ]:
cv = evaluation.make_cv()
linear_preprocessor, tree_preprocessor = processing.build_preprocessors(engineered_numerical_features, categorical_features)
scale_pos_weight = models.compute_scale_pos_weight(target_data_train)
candidate_models = models.build_models(tree_preprocessor, linear_preprocessor, scale_pos_weight)

print(f"Training-set scale_pos_weight: {scale_pos_weight:.3f}")
print(f"Candidate models: {list(candidate_models)}")

## Evaluation policy

- **Primary model-ranking metric:** cross-validated Average Precision
- **Secondary metrics:** ROC-AUC, precision, recall, F1, and F2
- **Threshold rule:** maximize precision while maintaining recall at or above `MIN_RECALL`
- **Final reporting:** Average Precision, ROC-AUC, precision, recall, F1, F2, balanced accuracy, confusion counts, and bootstrap uncertainty

The test set remains untouched until the final evaluation section.

# 8. Baseline Model Comparison

In [ ]:
cv_results = evaluation.cross_validate_models(candidate_models, feature_data_train, target_data_train, cv)
display(cv_results.round(4))
reporting.save_table(cv_results, tables_dir / "cross_validation_results.csv", include_index=False)

In [ ]:
cv_figure = plotting.plot_cv_comparison(cv_results)
display(cv_figure)
plotting.save_figure(cv_figure, figures_dir / "cross_validation_comparison.png")

Select tuning candidates using the observed cross-validation ranking rather than expected model behavior. This repository tunes Random Forest and XGBoost because they are the designated nonlinear candidates. The results should explicitly state whether either actually outperformed the simpler baselines.

# 9. Hyperparameter Tuning

In [ ]:
rf_search = models.tune_random_forest(candidate_models, feature_data_train, target_data_train, cv)
print("Best Random Forest parameters:")
print(rf_search.best_params_)
print(f"Best Random Forest CV Average Precision: {rf_search.best_score_:.4f}")

In [ ]:
xgb_search = models.tune_xgboost(candidate_models, feature_data_train, target_data_train, cv)
print("Best XGBoost parameters:")
print(xgb_search.best_params_)
print(f"Best XGBoost CV Average Precision: {xgb_search.best_score_:.4f}")

In [ ]:
baseline_scores = cv_results.set_index("Model")["CV Average Precision"]
tuning_comparison = pd.DataFrame([{"Model": "Random Forest", "Untuned CV Average Precision": baseline_scores["Random Forest"], "Tuned CV Average Precision": rf_search.best_score_}, {"Model": "XGBoost", "Untuned CV Average Precision": baseline_scores["XGBoost"], "Tuned CV Average Precision": xgb_search.best_score_}])
tuning_comparison["Improvement"] = tuning_comparison["Tuned CV Average Precision"] - tuning_comparison["Untuned CV Average Precision"]

display(tuning_comparison.round(4))
reporting.save_table(tuning_comparison, tables_dir / "tuning_comparison.csv", include_index=False)

A negative tuning improvement is a legitimate result. It means the searched configuration did not outperform the existing baseline under the selected cross-validation design.

In [ ]:
final_model_name, final_model = models.select_final_model({"Tuned Random Forest": rf_search, "Tuned XGBoost": xgb_search})
print(f"Selected final model among tuned candidates: {final_model_name}")

The selection above is based only on training-set cross-validation scores. The estimator returned by `RandomizedSearchCV` is refitted on the full training set and remains isolated from the test labels.

# 10. Threshold Selection, Calibration, and Operational Cost

In [ ]:
cv_probabilities = evaluation.get_oof_probabilities(final_model, feature_data_train, target_data_train, cv)
final_threshold, threshold_results, (precision, recall, thresholds) = evaluation.select_threshold(target_data_train, cv_probabilities)

print(f"Recall-constrained threshold: {final_threshold:.4f}")
display(threshold_results.head(10).round(4))
reporting.save_table(threshold_results, tables_dir / "threshold_results.csv", include_index=False)

In [ ]:
threshold_figure = plotting.plot_threshold_tradeoff(thresholds, precision, recall, final_threshold)
display(threshold_figure)
plotting.save_figure(threshold_figure, figures_dir / "threshold_tradeoff.png")

## Calibration

Calibration is necessary when model outputs are interpreted as failure probabilities rather than only ranking scores.

In [ ]:
fraction_of_positives, mean_predicted_value = evaluation.get_calibration_curve(target_data_train, cv_probabilities)
calibration_table = pd.DataFrame({"Mean predicted probability": mean_predicted_value, "Observed failure rate": fraction_of_positives})

display(calibration_table.round(4))
reporting.save_table(calibration_table, tables_dir / "calibration_curve.csv", include_index=False)

In [ ]:
calibration_figure = plotting.plot_calibration(mean_predicted_value, fraction_of_positives, final_model_name)
display(calibration_figure)
plotting.save_figure(calibration_figure, figures_dir / "calibration_curve.png")

## Illustrative expected-cost analysis

The configured false-negative and false-positive costs are scenario assumptions, not measured business values. The cost-minimizing threshold is shown as a sensitivity analysis. The recall-constrained threshold remains the primary operating threshold unless stakeholders validate the cost inputs.

In [ ]:
cost_curve, cost_minimizing_threshold = evaluation.expected_cost_curve(target_data_train, cv_probabilities, thresholds)
print(f"Cost-minimizing threshold under configured assumptions: {cost_minimizing_threshold:.4f}")

display(cost_curve.sort_values("Expected Cost").head(10).round(3))
reporting.save_table(cost_curve, tables_dir / "expected_cost_curve.csv", include_index=False)

In [ ]:
cost_figure = plotting.plot_expected_cost(cost_curve, final_threshold, cost_minimizing_threshold)
display(cost_figure)
plotting.save_figure(cost_figure, figures_dir / "expected_cost_curve.png")

# 11. Final Test Evaluation

The test set is first exposed in this section. Afterward it is used only for descriptive diagnostics of the frozen model; no feature, model, hyperparameter or threshold is changed.

In [ ]:
test_probability, test_prediction, final_test_metrics = evaluation.evaluate_on_test(final_model, final_model_name, final_threshold, feature_data_test, target_data_test)
final_metrics_table = pd.DataFrame([final_test_metrics]).rename(columns={"PR-AUC": "Average Precision"})

display(final_metrics_table.round(4))
reporting.save_table(final_metrics_table, tables_dir / "final_test_metrics.csv", include_index=False)

In [ ]:
classification_table = reporting.classification_report_table(target_data_test, test_prediction)
display(classification_table.round(4))
reporting.save_table(classification_table, tables_dir / "classification_report.csv")

In [ ]:
confusion_figure = plotting.plot_confusion_matrix(target_data_test, test_prediction, final_model_name)
display(confusion_figure)
plotting.save_figure(confusion_figure, figures_dir / "confusion_matrix.png")

In [ ]:
roc_figure = plotting.plot_roc_curve(target_data_test, test_probability, final_model_name)
display(roc_figure)
plotting.save_figure(roc_figure, figures_dir / "roc_curve.png")

In [ ]:
pr_figure = plotting.plot_precision_recall_curve(target_data_test, test_probability, final_model_name)
display(pr_figure)
plotting.save_figure(pr_figure, figures_dir / "precision_recall_curve.png")

Read the four artifacts above together, not independently. The metrics table and classification report provide point estimates at the fixed threshold, while the confusion matrix illustrates the underlying raw data. The ROC and Precision-Recall curves demonstrate performance across every threshold, not just the selected one. This makes them valuable for determining whether the chosen operating point is close to the optimal trade-off or if a significantly better point exists elsewhere on the curve.

As Average Precision, rather than ROC-AUC, is the model-selection metric used throughout this notebook (see Evaluation Policy, Section 7), give more weight to the Precision-Recall curve than to the ROC curve when judging how much headroom is left. ROC-AUC is reported for reference only, as under this level of class imbalance, it is the less informative of the two summaries.

## Bootstrap confidence intervals

The test set contains relatively few failures, so point estimates can vary substantially. Percentile bootstrap intervals communicate this uncertainty more honestly than reporting metrics alone.

In [ ]:
bootstrap_metrics = evaluation.bootstrap_confidence_intervals(target_data_test, test_probability, final_threshold)
ci_table = reporting.confidence_interval_table(bootstrap_metrics, final_test_metrics)
ci_display = ci_table.rename(index={"PR-AUC": "Average Precision"})

display(ci_display.round(4))
reporting.save_table(ci_display, tables_dir / "bootstrap_confidence_intervals.csv")

# 12. Interpretability and Error Analysis

These analyses explain the already-frozen model. They must not feed back into model selection.

## Permutation importance

Permutation importance measures the drop in held-out Average Precision when a single feature's values are randomly shuffled, holding everything else fixed. Because it is computed directly against a held-out performance metric rather than an internal split-count heuristic, it is treated as the primary importance signal in this notebook. The native (impurity- or gain-based) importance in the next subsection is reported for comparison but is secondary, see the note there for why the two can disagree.

In [ ]:
permutation_importance_df = interpretability.permutation_importance_table(final_model, feature_data_test, target_data_test)
display(permutation_importance_df.sort_values("Importance Mean", ascending=False).round(4))
reporting.save_table(permutation_importance_df, tables_dir / "permutation_importance.csv", include_index=False)

In [ ]:
permutation_figure = plotting.plot_permutation_importance(permutation_importance_df, final_model_name)
display(permutation_figure)
plotting.save_figure(permutation_figure, figures_dir / "permutation_importance.png")

## Native tree-model importance

Native importance is model-specific and can be biased toward continuous or high-cardinality features. It is therefore secondary to permutation importance.

In [ ]:
native_importance_df = interpretability.native_importance_table(final_model)
display(native_importance_df.sort_values("Importance", ascending=False).round(4))
reporting.save_table(native_importance_df, tables_dir / "native_importance.csv", include_index=False)

In [ ]:
native_figure = plotting.plot_native_importance(native_importance_df, final_model_name)
display(native_figure)
plotting.save_figure(native_figure, figures_dir / "native_importance.png")

## Product-type slice performance

Always interpret slice metrics together with sample counts and failure prevalence. A high recall based on very few positive cases is unstable.

In [ ]:
slice_performance_df = interpretability.slice_performance(feature_data_test, target_data_test, test_prediction, test_probability)
display(slice_performance_df.round(4))
reporting.save_table(slice_performance_df, tables_dir / "slice_performance.csv", include_index=False)

## Failure-mode detection

A row may contain more than one active failure mode, so mode-specific case counts can overlap.

In [ ]:
failure_mode_breakdown_df = interpretability.failure_mode_breakdown(maintenance_df, feature_data_test, test_prediction)
display(failure_mode_breakdown_df.round(4))
reporting.save_table(failure_mode_breakdown_df, tables_dir / "failure_mode_breakdown.csv", include_index=False)

## False negatives and false positives

False negatives deserve particular attention because the operating objective prioritizes failure detection. TWF cases may be only partially predictable from tool wear, while RNF cases are not recoverable from the available operating inputs.

In [ ]:
error_analysis = interpretability.build_error_analysis(feature_data_test, target_data_test, test_probability, test_prediction)
false_negatives = interpretability.false_negatives_table(error_analysis, maintenance_df)
false_positives = interpretability.false_positives_table(error_analysis)

display(false_negatives.head(10))
display(false_positives.head(10))

reporting.save_table(error_analysis, tables_dir / "error_analysis.csv")
reporting.save_table(false_negatives, tables_dir / "false_negatives.csv")
reporting.save_table(false_positives, tables_dir / "false_positives.csv")

In [ ]:
error_group_medians = error_analysis.groupby(["Actual", "Predicted"])[list(numerical_features) + ["Power", "Temperature difference", "Torque x Tool wear"]].median()
display(error_group_medians.round(3))
reporting.save_table(error_group_medians, tables_dir / "error_group_medians.csv")

# 13. Save the Final Model and Metadata

In [ ]:
model_path, metadata_path = reporting.save_model_artifacts(final_model, final_model_name, final_threshold, cost_minimizing_threshold, final_test_metrics, artifacts_dir, config.RANDOM_STATE, config.TEST_SIZE)

print(f"Model saved to: {model_path}")
print(f"Metadata saved to: {metadata_path}")

# 14. Conclusion

In [ ]:
reporting.print_final_summary(final_model_name, final_threshold, final_test_metrics, ci_table)

## Final Interpretation

The **Tuned Random Forest** was selected as the final model with a cross-validated Average Precision of **0.8937**. Tuning produced only a marginal improvement over the untuned Random Forest (**0.8931 → 0.8937**), while tuned XGBoost improved more noticeably (**0.8651 → 0.8812**) but remained weaker.

At the recall-constrained threshold of **0.575**, the model achieved a test Average Precision of **0.876**, recall of **0.794**, precision of **0.947** and F2-score of approximately **0.821**. It detected **54 of 68 failures**, with **14 false negatives** and only **3 false positives**. Bootstrap intervals show uncertainty in Average Precision (**0.798–0.942**), recall (**0.687–0.886**) and precision (**0.881–1.000**).

`Rotational speed` was the strongest feature in both importance analyses. The engineered features `Temperature difference`, `Torque x Tool wear` and `Power` also contributed substantially, supporting the feature-engineering decisions.

Performance varied by product type. Type H had the weakest recall (**0.600**) and precision (**0.750**), although it contained only five failures. Type M achieved the lowest slice Average Precision (**0.841**), while Type L showed more balanced performance.

Failure-mode recall was strongest for OSF (**1.000**), HDF (**0.931**) and PWF (**0.923**). The model performed poorly on TWF, detecting only **1 of 10 cases** and missed all **4 RNF cases**. These small subgroup sizes make the estimates uncertain but they identify the main areas for further error analysis.

The cost-minimizing threshold of **0.105** was substantially lower than the recall-constrained threshold of **0.575**, indicating that the assumed cost function favors a more aggressive strategy with more false alarms to reduce missed failures. The final threshold should therefore depend on the operational cost of inspections relative to undetected failures.

These findings are specific to the synthetic AI4I benchmark and do not demonstrate real-world deployment performance.

## Limitations

- **Synthetic benchmark:** results do not establish production performance.
- **Random stratified holdout:** neighboring sequential operating states may appear across train and test; a chronological holdout should be added as a robustness experiment.
- **Limited positive test cases:** failure-class estimates have material uncertainty.
- **Threshold-selection optimism:** hyperparameters and the threshold are selected from the same overall training set, although threshold predictions are out of fold. A dedicated validation split or nested CV would provide stricter separation.
- **Illustrative costs:** false-positive and false-negative costs must be validated by domain stakeholders.
- **Calibration:** poor calibration would require a calibrated classifier before treating scores as literal probabilities.
- **Failure-rule recoverability:** because several synthetic failure rules are deterministic functions of the inputs, this benchmark partly measures recovery of known generation mechanisms.